# Numerical parameter identifiability

See the [course by Marisa Eisenberg](https://epimath.org/epid-814-materials/) for an excellent introduction into the topic


<img src="assets/identifiability.png" style="max-height: 300px;">

In [ ]:
import numpy as np

from mxlpy import Model, Simulator, fit, fns, identify, plot

We start with an SIR model, which we use to generate some data (this would usually be experimentally measured data)

In [ ]:
def sir() -> Model:
    return (
        Model()
        .add_variables({"s": 0.9, "i": 0.1, "r": 0.0})
        .add_parameters({"beta": 0.2, "gamma": 0.1})
        .add_reaction(
            "infection",
            fns.mass_action_2s,
            args=["s", "i", "beta"],
            stoichiometry={"s": -1, "i": 1},
        )
        .add_reaction(
            "recovery",
            fns.mass_action_1s,
            args=["i", "gamma"],
            stoichiometry={"i": -1, "r": 1},
        )
    )


data = Simulator(sir()).simulate(100).get_result().unwrap_or_err().variables
_ = plot.lines(data)

## Profile likelihood

The profile likelihood of a parameter fixes it at values stepping away from the optimum and refits all other parameters at each of them.
The increase of the objective along the profile shows how well the data determines the parameter:

- with a likelihood loss like `fit.GaussianNll(sigma)`, the likelihood-ratio test bounds the `alpha` confidence interval where the objective has increased by `chi2.ppf(alpha, 1) / 2`
- a profile that crosses this threshold on both sides is **identifiable**
- one that reaches the end of the profiled range on a side without crossing it is **practically non-identifiable**, the data only bounds it on one side, if at all
- a flat profile is **structurally non-identifiable**, other parameters fully compensate for it

Steps adapt to the profile, dense near the optimum and sparse where it is flat, and both sides of all parameters are profiled in parallel.
With `transform="log"` parameters are profiled in log10 steps, which suits values spanning orders of magnitude.
For losses that aren't likelihoods, pass the threshold yourself.

We add measurement noise to the data, fit the model and profile both parameters from the optimum.

In [ ]:
rng = np.random.default_rng(42)
noisy = data + rng.normal(0, 0.01, data.shape)
nll = fit.GaussianNll(sigma=0.01)

res = fit.time_course(
    sir(),
    p0={"beta": 0.3, "gamma": 0.05},
    data=noisy,
    loss_fn=nll,
    minimizer=fit.LeastSquaresMinimizer(),
).unwrap_or_err()

profiles = identify.profile_likelihood(sir(), noisy, res, loss_fn=nll, transform="log")
_ = plot.profile_likelihood(profiles)
profiles.ci

## Fisher information

Profile likelihoods need many fits per parameter.
A much cheaper first look is the Fisher information at the optimum, which needs a single [sensitivity](forward-sensitivities.ipynb) evaluation.
From the noise-weighted sensitivities of the residuals `S` it computes `F = SᵀS`, whose inverse is the asymptotic covariance of the parameters.
It assumes Gaussian noise and a model that is approximately linear around the optimum.

Pass the noise level as `sigma` (a scalar, one value per observable or per data point).
Without it, the noise variance is estimated from the residuals.

In [ ]:
fim = identify.fisher_information(
    sir(),
    data=data,
    parameters={"beta": 0.2, "gamma": 0.1},  # usually a fit result
    sigma=0.01,
)
fim.wald_ci(alpha=0.95)

The correlation matrix shows which parameters compensate for each other.

In [ ]:
fig, ax, _ = plot.heatmap(fim.correlation, annotate=True)
plot.show()

Small eigenvalues of `F` belong to *sloppy* parameter combinations, which the data barely constrains.
If we only observe `i` during the first two time units, it grows approximately with rate `0.9 * beta - gamma`.
Only this combination is determined: the eigenvalues differ by orders of magnitude, the eigenvector of the small one changes `beta` and `gamma` together in a ratio of about `1 : 0.9`, and the confidence intervals become huge.

In [ ]:
early = data.loc[:2, ["i"]]
fim = identify.fisher_information(
    sir(),
    data=early,
    parameters={"beta": 0.2, "gamma": 0.1},
    sigma=0.01,
)
eigenvalues, eigenvectors = fim.eigen()
eigenvalues

In [ ]:
eigenvectors

In [ ]:
fim.wald_ci()

If `F` is exactly singular (e.g. only the product of two parameters enters the model), `fim.non_identifiable()` returns the parameter combinations spanning its null space, and the standard errors of the parameters involved are infinite.
With `transform="log"` uncertainties are expressed in log10 units, i.e. relative to the parameter values.

## Fast identifiability test

Profiles need many fits per parameter, the Fisher information assumes the model is approximately linear around the optimum.
`identify.identifiability_test` is a cheap numerical check in between: it refits the parameters from `n_fits` random points around the optimum, with a penalty that pulls them to distance `radius` from it (in log10 units with `transform="log"`, the default, so `radius=1` is a decade).

- if a pulled fit reaches the radius without increasing the objective by more than the threshold (`chi2.ppf(0.95, 1) / 2` for likelihood losses, else `threshold_rtol` times the objective), the objective is flat in that direction
- fits that stop short of the radius without exceeding the threshold (usually because of bounds) make the result inconclusive, which the test warns about
- the parameters that moved along it (by at least `min_displacement · radius`) are non-identifiable

This costs a handful of fits regardless of the number of parameters, which makes it a good routine check after every fit.
It is a check, not a proof: flat directions the random starts miss go unnoticed, so follow up with profiles of the flagged parameters.

Both SIR parameters pass the test.

In [ ]:
test = identify.identifiability_test(sir(), noisy, res, loss_fn=nll)
test.is_identifiable, test.non_identifiable

Now we make the model over-parametrised: the infection rate is the product of a contact rate `c` and the transmission probability `beta`, which the data can't tell apart.
The test finds the flat direction, and `displacement` ranks the parameters by how far they moved along it.

In [ ]:
def sir_contacts() -> Model:
    return (
        Model()
        .add_variables({"s": 0.9, "i": 0.1, "r": 0.0})
        .add_parameters({"beta": 0.2, "gamma": 0.1, "c": 1.0})
        .add_derived("rate", fns.mul, args=["beta", "c"])
        .add_reaction(
            "infection",
            fns.mass_action_2s,
            args=["s", "i", "rate"],
            stoichiometry={"s": -1, "i": 1},
        )
        .add_reaction(
            "recovery",
            fns.mass_action_1s,
            args=["i", "gamma"],
            stoichiometry={"i": -1, "r": 1},
        )
    )


res_c = fit.time_course(
    sir_contacts(),
    p0={"beta": 0.3, "gamma": 0.05, "c": 1.5},
    data=noisy,
    loss_fn=nll,
    minimizer=fit.LeastSquaresMinimizer(),
).unwrap_or_err()

test = identify.identifiability_test(sir_contacts(), noisy, res_c, loss_fn=nll)
print(test.is_identifiable, test.non_identifiable)
test.displacement

The left plot shows the displacements, the right one each pulled fit's increase of the objective against its distance from the optimum: flat fits (red) reach the radius without crossing the threshold.

In [ ]:
_ = plot.identifiability_test(test)

With `recursive=True`, the test fixes the parameter that moved most at its optimal value and repeats until no flat direction is left.
`test.fixed` then lists parameters whose fixing makes the model identifiable, here one of `beta` and `c`.

In [ ]:
test = identify.identifiability_test(
    sir_contacts(), noisy, res_c, loss_fn=nll, recursive=True
)
test.fixed, [r.is_identifiable for r in test.rounds]

## Related parameters

Along the profile of a non-identifiable parameter, other parameters compensate for it so the fit stays the same.
`identify.related_parameters` regresses each such profile on the paths of the other parameters and returns the combination that stays constant, which is what the data actually determine.
With log parameters, `log10(beta) + log10(c) ≈ const` means that the product `beta · c` is identifiable.

In [ ]:
profiles = identify.profile_likelihood(
    sir_contacts(), noisy, res_c, loss_fn=nll, transform="log", names=["beta", "c"]
)
related = identify.related_parameters(profiles)
print(related.groups)
for relation in related.relations:
    print(relation)